# 08 - Serving: pushing a live signal without lying about it

Stage 7b produced an 11 us C++ forward pass. This notebook is about the layer
between that work and the only artefact most people will ever see, and about
the three ways a serving layer quietly misrepresents the system underneath it:
by showing stale data as current, by smoothing over a discontinuity that
matters, and by quoting the flattering latency number.

`serving/api.py` is a long-running server and the constitution exempts it from
notebook coverage. Everything it *uses* - `feeds.py`, `signal.py`,
`records.py` - is importable, and that is what runs here.

In [1]:
import asyncio, json, sys
from pathlib import Path

import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "serving").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from serving.feeds import BookAnchor, ReplayFeed, SessionBoundary, default_replay_tapes

feed = ReplayFeed(default_replay_tapes(REPO_ROOT), speed="max", loop=True)
described = feed.describe()
for session in described["sessions"]:
    print(f"  {session['name']}: {session['anchors']:5d} anchors  "
          f"{session['extent_s']:5.1f}s  {session['anchor_rate_hz']:5.2f} Hz")
print(f"\ntotal {described['total_extent_s']:.1f}s  ->  "
      f"{described['total_extent_s'] / 10:.1f}s loop at 10x")
print("tick size measured from the data:",
      described["tick_size_fixed"] / described["price_scale"])

  btcusdt_replay_s0:  1571 anchors   70.0s  22.44 Hz
  btcusdt_replay_s1:  1704 anchors   69.8s  24.40 Hz
  btcusdt_replay_s2:  2270 anchors   69.9s  32.48 Hz

total 209.7s  ->  21.0s loop at 10x
tick size measured from the data: 0.01


Three tapes, not one, and that is why this set exists.

The committed `btcusdt_dense_sample.tape` is a prefix of **one** capture
session, so it contains no session boundary at all - and at 65.8 s it loops
every 6.6 s at 10x, too fast to watch. These are ~70 s prefixes of all three
real sessions from 2026-07-27: 209.7 s, a 21 s loop at 10x, crossing a genuine
resync every 7 s.

The tick size is **measured**, never written down. `measure_tick_size` takes
the smallest positive gap between adjacent price levels across every snapshot;
on a book this dense that gap is the venue tick. It comes out at 0.01 USDT,
correct for BTCUSDT - and it would still be correct pointed at a symbol whose
tick is not 0.01.

## 1. A session boundary is an event, not a gap to paper over

The capture daemon starts a **new tape file** when it resyncs, so session
boundaries live between files rather than inside them. That is why the feed
yields a union: a consumer has to handle `SessionBoundary` to make sense of the
stream at all.

Stage 5 built `resolve_prices_per_session` to keep returns from being computed
across those gaps. A serving feed that silently re-joined what the research
code carefully split would put that error back where nobody is looking.

In [2]:
async def take(source, limit):
    events = []
    async for event in source.events():
        events.append(event)
        if len(events) >= limit:
            return events

events = await take(feed, 12000)  # noqa: F704 - ipykernel allows top-level await
anchors = [e for e in events if isinstance(e, BookAnchor)]
boundaries = [e for e in events if isinstance(e, SessionBoundary)]
print(f"{len(anchors)} anchors, {len(boundaries)} boundaries\n")
for b in boundaries:
    print(f"  {b.reason:7s}  {b.session_id:22s} <- {str(b.previous_session_id):22s} "
          f"gap {b.gap_ns / 1e9:7.1f}s")

11993 anchors, 7 boundaries

  start    btcusdt_replay_s0#0    <- None                   gap     0.0s
  resync   btcusdt_replay_s1#0    <- btcusdt_replay_s0#0    gap   422.5s
  resync   btcusdt_replay_s2#0    <- btcusdt_replay_s1#0    gap   543.8s
  loop     btcusdt_replay_s0#1    <- btcusdt_replay_s2#0    gap     0.0s
  resync   btcusdt_replay_s1#1    <- btcusdt_replay_s0#1    gap   422.5s
  resync   btcusdt_replay_s2#1    <- btcusdt_replay_s1#1    gap   543.8s
  loop     btcusdt_replay_s0#2    <- btcusdt_replay_s2#1    gap     0.0s


Two resyncs and a loop wrap per cycle, each announced *before* any anchor
carries the new id. A test asserts that invariant, because it is what lets a
consumer know when to throw its history away.

**The gaps read 422.5 s and 543.8 s, not the 9.1 s and 9.4 s that separated the
full captures.** That is correct rather than a bug: the gap is measured between
the last anchor actually present in one tape and the first in the next, so it
includes the ~413 s of session 0 that truncation removed *as well as* the real
resync. It is the number a consumer needs - how much market this replay did not
observe - and a return computed across it would be wrong for both reasons.

A loop wrap reports `gap_ns = 0`. Its discontinuity is an artefact of looping,
and inventing a duration for it would be a fabrication.

## 2. The 599-anchor blackout, and why it is shown rather than hidden

`ml/features.py` normalises with a **causal** rolling z-score over 500 rows,
and the model window is 100 rows. One model input therefore needs
`100 + 500 - 1 = 599` anchors of history - and every session boundary destroys
that history on purpose.

So after every resync, loop wrap and seek there are 599 anchors with **no
signal at all**: roughly 20 s at 1x, 2 s at 10x. The temptation is to keep
showing the last good signal. That would be exactly the dishonesty this project
exists to avoid - a confident arrow on screen corresponding to no live input.

In [3]:
from serving.signal import HISTORY_ROWS, SignalEngine

engine = SignalEngine(REPO_ROOT / "notebooks" / "sample_data" / "student_int8.onnx",
                      feed.price_scale, feed.qty_scale)
print("history rows required:", HISTORY_ROWS)

for index, anchor in enumerate(anchors[:HISTORY_ROWS + 2]):
    result = engine.push(anchor.bids, anchor.asks)
    if index in (0, 100, 597, 598, 599, 600):
        state = ("ready" if result.ready
                 else f"warming, {result.anchors_until_ready} anchors to go")
        print(f"  anchor {index:4d}: {state}")

print("\nfirst signal:",
      {name: round(value, 5)
       for name, value in zip(("p_down", "p_flat", "p_up"), result.probabilities)})

history rows required: 599
  anchor    0: warming, 598 anchors to go


  anchor  100: warming, 498 anchors to go
  anchor  597: warming, 1 anchors to go
  anchor  598: ready
  anchor  599: ready
  anchor  600: ready

first signal: {'p_down': 0.99987, 'p_flat': 0.00012, 'p_up': 0.0}


The engine reports `ready=False` and a countdown, and the websocket frame
carries both as `signal: null` plus a `warmup` object. A dashboard can render
"warming up, 412 anchors to go" - which is true - instead of a stale arrow,
which is not.

This is also why `reset()` is wired to the boundary rather than to a timer.
Carrying the ring across a resync would normalise one session's book against
another session's statistics, across a gap containing price moves nobody saw.

## 3. The latency boundary, measured rather than assumed

`serving_infer_us` in every frame covers the ONNX forward pass and nothing
else. Feature construction is timed separately and never folded in.

I originally wrote a docstring guessing that feature construction would be
negligible next to the model. The guess was wrong, and measuring is the only
reason I know that.

In [4]:
infer_ns, feature_ns = [], []
for anchor in anchors[HISTORY_ROWS:HISTORY_ROWS + 400]:
    result = engine.push(anchor.bids, anchor.asks)
    if result.ready:
        infer_ns.append(result.infer_ns)
        feature_ns.append(result.feature_ns)

for name, samples in (("ONNX int8 forward pass", infer_ns),
                      ("feature construction", feature_ns)):
    p50, p99 = np.percentile(samples, [50, 99])
    print(f"  {name:24s} p50 {p50 / 1000:8.1f} us   p99 {p99 / 1000:8.1f} us")

share = np.median(feature_ns) / (np.median(infer_ns) + np.median(feature_ns))
print(f"\nfeature construction is {share:.0%} of the per-anchor cost")
print(f"clock resolution {engine.clock.resolution_ns} ns; can resolve this:",
      engine.clock.can_resolve(np.median(infer_ns)))

  ONNX int8 forward pass   p50    576.1 us   p99   1225.1 us
  feature construction     p50    352.9 us   p99    976.2 us

feature construction is 38% of the per-anchor cost
clock resolution 100 ns; can resolve this: True


Feature construction is roughly **40% of the per-anchor cost** - not a rounding
error, and not something to hide inside one number.

This makes the C++ comparison *more* lopsided than it looks. Stage 7b's ~11 us
covers a hand-written forward pass from an **already-prepared** `[40]` feature
column; it excludes all of the work in the second row above, and no C++ feature
path exists at all (`TODO(measure)` in the methodology). `/pareto` tags every
row with `measured_by` for exactly this reason, and `/latency` refuses to
publish percentiles the clock cannot resolve - the check that caught
`steady_clock` recording 19,518 of 20,000 C++ passes as zero in Stage 7b.

## 4. Drop, never queue - and the bug that hid underneath it

A stale frame is not a worse frame, it is a **wrong** one: it draws a book that
no longer exists beside a signal about a price that has already moved, and the
viewer cannot tell. So `ClientChannel` holds exactly one pending frame and the
older one dies.

That was not enough, and only running the container proved it.

In [5]:
from serving.api import ClientChannel

mailbox = ClientChannel()
for index in range(50):
    mailbox.offer(f"frame-{index}")
print(f"fire-and-forget: client gets {await mailbox.take()} "
      f"({mailbox.dropped_frames} dropped)")

async def credited():
    channel = ClientChannel(credit=1)
    channel.offer("frame-0")
    first = await channel.take()
    for index in range(1, 50):
        channel.offer(f"frame-{index}")
    blocked = asyncio.create_task(channel.take())
    await asyncio.sleep(0)
    waiting = not blocked.done()
    channel.grant()
    return first, waiting, await blocked

first, waiting, second = await credited()
print(f"credit: sent {first}, blocked without ack: {waiting}, then {second}")

fire-and-forget: client gets frame-49 (49 dropped)
credit: sent frame-0, blocked without ack: True, then frame-49


The mailbox is correct and was insufficient. `websocket.send_text()` returns
when the frame reaches the **transport**, not the client, and uvicorn has a
write buffer with a socket buffer under it.

Measured against the running container - a client reading one frame every
250 ms while the server produced ~244 frames/s:

| | sequence numbers advanced per read |
|---|---|
| expected if fresh | ~60 |
| mailbox only | **1** |
| `?flow=ack` | **34 median, 78 max** |

The server had accepted 239 sends for 31 receives: ~208 stale frames buffered
below the mailbox and delivered *in order*, minutes behind the market. **The
unit test passed the whole time**, because it tested the component and the bug
was underneath it. Credit mode hands the transport at most one unacknowledged
frame, so there is nothing left to buffer.

### 4b. A boundary is not state - and the same mailbox ate every one of them

The drop policy above is right about *state*. Stage 8b found it was also being
applied to *events*, which is a different thing entirely, and the consequence
was the worst class of bug this project has.

In [6]:
channel = ClientChannel()
channel.post("session_boundary")          # an EVENT: not superseded by anything
for index in range(100):                  # 100 newer frames arrive behind it
    channel.offer(f"frame-{index}")

print("first message out:", await channel.take())
print("then:", await channel.take())
print("frames dropped:", channel.dropped_frames, " events dropped:", channel.dropped_events)

first message out: session_boundary
then: frame-99
frames dropped: 99  events dropped: 0


Session boundaries used to be broadcast through `offer`, so they shared the
one-frame slot with book state. The producer emits ~250 frames a second and a
render-driven client takes 60, so the anchor arriving immediately after a
boundary overwrote it essentially every time.

Measured against the running server with a credit-mode client, before the fix:

| 30 seconds of stream | delivered |
|---|---|
| frames | 1,126 |
| session boundaries | **0** |

Roughly five real boundaries occurred in that window. The client was never told
that its history had been invalidated, so the Stage 8b dashboard drew one
continuous tape straight across a **422-second hole in the data** - which is
precisely the fabrication `serving/feeds.py` emits boundaries as events to
prevent, undone one layer further out.

The distinction that was missing is the whole fix. A book frame is **state**: a
newer one makes an older one worthless, so dropping it is correct. A boundary is
an **event**: nothing supersedes it, it says everything before it must be
discarded, and delivering it late is still useful while not delivering it at all
is corrupting. State gets a one-slot mailbox; events get a bounded queue that is
drained first. `dropped_events` is reported by `/health` and should be zero
forever.

Note also what found it. Not a test - the unit tests all passed, again - but
building a client that actually *used* the boundary for something visible.

## 5. Every served number comes out of a file

Stage 6 measured 55% run-to-run latency variance on this machine. A constant
typed into the serving layer would therefore not be a shortcut - it would be a
slowly-rotting lie nobody notices, because the dashboard keeps rendering.

So `serving/records.py` reads `benchmarks/*.json` at request time, and every
payload names the file it came from.

In [7]:
from serving import records

economics = records.economics_payload()
print("source:", economics["source"])
print(f"breakeven fee {economics['breakeven_fee_bps']:.3f} bps per side\n")
for tier in economics["tiers"]:
    print(f"  {tier['name']:14s} {tier['fee_bps']:5.1f} bps  "
          f"{tier['shortfall_multiple']:5.1f}x short  "
          f"net {tier['net_bps_per_trade']:+.3f} bps/trade")

stability = records.stability_payload()
print("\nkeys /stability offers containing 'pooled':",
      [key for key in stability if "pooled" in key])
print(f"per-block mean IC {stability['mean']:+.4f}, "
      f"IR {stability['information_ratio']:.2f}")
print(f"pooled            {stability['pooled_ic_not_tradeable']:+.4f}")

source: evaluation_20260728T202537Z.json
breakeven fee 0.142 bps per side

  VIP 0           10.0 bps   70.2x short  net -19.715 bps/trade
  VIP 0 + BNB      7.5 bps   52.6x short  net -14.715 bps/trade
  VIP 4            5.0 bps   35.1x short  net -9.715 bps/trade
  VIP 9            4.0 bps   28.1x short  net -7.715 bps/trade

keys /stability offers containing 'pooled': ['pooled_ic_not_tradeable', 'pooled_caveat']
per-block mean IC +0.0726, IR 0.21
pooled            +0.4209


Every published Binance taker tier is unaffordable - 28x to 70x short - and a
test asserts every tier's net edge is negative, so if the record ever moved
enough to make one affordable the suite would say so rather than the dashboard
quietly implying it.

Note the key name. There is no `pooled_ic` field anywhere in `/stability`; the
pooled figure is served as **`pooled_ic_not_tradeable`**. CLAUDE.md forbids
presenting +0.421 as the edge, and naming is the cheapest structural guard
available: a dashboard author reaching for the biggest number has to type the
words "not tradeable" to get it. That is a constraint by construction rather
than by care, which is the only kind that survives a deadline.

## 6. The frontier carries both C++ rows, and neither claims a measured accuracy

Stage 8b's dashboard needs six labelled points on one axis. `/pareto` was
serving five, because `records.py` picked only the incremental C++ run.

In [8]:
pareto = records.pareto_payload()
for row in pareto["rows"]:
    inherited = " (inherited)" if row.get("macro_f1_is_inherited") else ""
    print(f"{row['label']:24s} p50 {row['p50_us']:9.1f} us   "
          f"F1 {row['macro_f1']:.4f}{inherited:12s} {row['measured_by']}")

print("\nsources:", json.dumps(pareto["sources"], indent=2))

PyTorch eager fp32       p50   11186.0 us   F1 0.5493             python harness
ONNX fp32                p50    1718.4 us   F1 0.5493             python harness
ONNX int8                p50    1147.2 us   F1 0.5469             python harness
distilled student fp32   p50     297.1 us   F1 0.5954             python harness
distilled student int8   p50     729.4 us   F1 0.5723             python harness
C++ full                 p50    2118.5 us   F1 0.5954 (inherited) cpp harness
C++ incremental          p50      11.0 us   F1 0.5954 (inherited) cpp harness

sources: {
  "python": "python_variants_20260801T082226Z.json",
  "cpp_full": "cpp_full_p4_stream.json",
  "cpp_incremental": "cpp_incremental_p4_stream.json"
}


Both C++ rows are served now, and the gap between them is the entire Stage 7b
finding. The incremental path is the one that ships at ~11 us; the **full** pass
recomputes the whole 100-row window every tick and lands at ~2,119 us - slower
than ONNX Runtime on the same model, because ONNX Runtime ships vectorised
kernels a hand-written nested loop does not match. Serving only the fast row
would present a ~193x *algorithmic* win as though it were what writing the thing
in C++ bought.

The fix also corrected a real error. The C++ row previously inherited
`student_int8`'s macro-F1 and was named `..._int8_equivalent` - but
`inference_cpp` is **float32**: its weights come from `student_distilled.pt`
with BatchNorm folded and nothing quantised anywhere. Crediting a float32
program with a quantised model's accuracy loss put those points 0.023 macro-F1
too low. The chain is fp32 student -> C++ full (parity, 1,000 windows, 2.4e-05)
-> C++ incremental (equivalence test), so `student_fp32` is what they inherit,
and `macro_f1_inherited_from` says so on every row.

Inherited is still not measured. No C++ program in this repository has ever
scored a test block, and the payload never pretends otherwise.

## Interview checkpoint

**1. Why push rather than poll, when the book only updates ~25 times a second?**
Because polling puts the *client* in charge of what "current" means. A client
that stalls for 300 ms comes back, asks for now, gets it, and silently skips
everything in between with no counter recording it. Push makes the server the
authority and makes the skipping explicit, countable and reported in `/health`.

**2. Your drop policy loses data. How do you justify that to a trader?**
By pointing out that the alternative is delivering it late, which is worse. A
frame that arrives stale is indistinguishable on screen from a current one, so
late delivery is a silent error where a dropped frame is a visible one. What
survives is always the newest state of the world, which is what a market view
exists to show.

**3. Your backpressure test passed while backpressure was broken. What went
wrong, and what is the general lesson?**
The test exercised `ClientChannel`, and the bug was underneath it in uvicorn's
write buffer - `send_text()` returns on hand-off, not on delivery - so a slow
client received strictly consecutive frames minutes late. The lesson is that a
test of a component is not a test of a system, and that claims about behaviour
under load have to be measured under load, in the real deployment.

**4. Why does the demo show nothing for the first two seconds after every
boundary?**
Because the causal rolling z-score needs 500 rows and the model window 100, so
599 anchors must arrive before an input exists - and a boundary destroys that
history deliberately, since normalising one session against another's
statistics across an unobserved gap is the error Stage 5 built machinery to
prevent. The frame reports `signal: null` and a countdown rather than repeating
a stale value.

**5. The C++ path is 11 us and this serves at ~1.1 ms. Why is the dashboard not
running the fast one?**
Because they are not the same measurement. The 11 us figure is a forward pass
from an already-prepared feature column, measured by a pinned harness over a
million iterations; it excludes feature construction, which is ~40% of the cost
here and has no C++ implementation at all. Beyond that, the honest answer is
that latency is not this signal's problem: the IC half-life is 13.2 s and the
fee shortfall is 70x, so a microsecond path buys nothing a millisecond path
does not. It is the prerequisite for signals with sub-second half-lives, and
this is not one.

**6. A dashboard is a client. What did building one teach you about the server
you had already finished and tested?**
That the one-frame mailbox was being applied to two different kinds of message.
Frames are state and are correctly dropped; session boundaries are events and
were being destroyed by the very next anchor, so in 30 seconds a client received
1,126 frames and zero boundaries. Nothing in the API's own test suite could see
it, because the component behaved exactly as specified - the specification was
wrong about one of its inputs. It took a consumer that had to *draw* the
boundary for the absence to become visible.

**7. Why does the dashboard's Pareto panel draw two different marks?**
Because the Python rows and the C++ rows come from different instruments
measuring different programs: the C++ harness times a forward pass from an
already-prepared `[40]` feature column and excludes the feature construction
that is inside every Python row. Filled circles are the Python harness and open
squares are the C++ harness, the boundary is stated permanently beneath the
panel, and the two C++ points are dimmed because their accuracy is inherited
rather than measured. Putting all seven on one axis without that distinction
would be an accusation waiting to happen.
